In [ ]:

# Standard Library
import os
import re
import json
import time
import uuid
import sqlite3
import hashlib
import logging
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict
from datetime import datetime

# Environment Variables
from dotenv import load_dotenv

# Numerical Computing
import numpy as np

# Data Processing
import pandas as pd

# NLP
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector Search
import faiss

# Knowledge Graph
import networkx as nx

# LLM Client (OpenAI SDK)
from openai import OpenAI

# Retry Logic
from tenacity import retry, stop_after_attempt, wait_exponential

# Progress Bars
from tqdm import tqdm


from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List
import os
from dotenv import load_dotenv

# CONFIGURATION

load_dotenv()


@dataclass
class Config:
    NVIDIA_API_KEY: str = os.getenv("NVIDIA_API_KEY")
    BASE_URL: str = "https://integrate.api.nvidia.com/v1"

    LLM_MODEL: str = "meta/llama-3.3-70b-instruct"
    EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"
    SPACY_MODEL: str = "en_core_web_sm"

    DATABASE_PATH: Path = Path("memory_ai.db")
    FAISS_INDEX_PATH: Path = Path("memory_index.faiss")
    GRAPH_PATH: Path = Path("memory_graph.pkl")
    LOG_PATH: Path = Path("memory.log")

    EMBEDDING_DIMENSION: int = 384

    MAX_NOTE_LENGTH: int = 10000
    MIN_NOTE_LENGTH: int = 20

    SUMMARY_MAX_WORDS: int = 120
    MAX_KEYWORDS: int = 15
    MAX_ENTITIES: int = 50
    MAX_RELATIONSHIPS: int = 50

    IMPORTANCE_THRESHOLD: float = 0.50
    DUPLICATE_THRESHOLD: float = 0.95
    SEARCH_SIMILARITY_THRESHOLD: float = 0.60

    TOP_K_VECTOR_RESULTS: int = 10
    TOP_K_KEYWORD_RESULTS: int = 10
    TOP_K_GRAPH_RESULTS: int = 10

    TEMPERATURE: float = 0.2
    MAX_TOKENS: int = 2048
    RETRY_ATTEMPTS: int = 3
    REQUEST_TIMEOUT: int = 60

    PROMPTS: Dict[str, str] = field(default_factory=lambda: {
        "summary": "Summarize the note while preserving all important facts.",
        "memory_extraction": "Extract structured memories as valid JSON.",
        "consolidation": "Merge related memories into higher-level memories.",
        "analysis": "Analyze memories for goals, skills, beliefs, interests, habits, personality, contradictions and predictions.",
        "qa": "Answer only using the retrieved memories. If unsupported, explicitly say so."
    })


config = Config()


def load_config():
    return config


def get_prompt(name: str):
    return config.PROMPTS[name]